In [5]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
import random

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0,
    # other params...
)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
import os

llmGoogle = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    api_key =os.getenv("google_api_key")
)

from langchain_core.messages import HumanMessage, SystemMessage

#  Simple llm app

## Using Langugage models

In [2]:
llm.invoke("Hello!")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-02-08T16:29:03.7487314Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9571729178, 'load_duration': 9218368522, 'prompt_eval_count': 12, 'prompt_eval_duration': 76278519, 'eval_count': 10, 'eval_duration': 267768636, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019c3e15-ae14-7970-b561-b12708069ee9-0', usage_metadata={'input_tokens': 12, 'output_tokens': 10, 'total_tokens': 22})

In [3]:
llm.invoke([{"role": "user","content":"Hello"}])

AIMessage(content='Hello! How are you today? Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-02-08T16:29:04.559355516Z', 'done': True, 'done_reason': 'stop', 'total_duration': 800014564, 'load_duration': 105578078, 'prompt_eval_count': 11, 'prompt_eval_duration': 62257347, 'eval_count': 23, 'eval_duration': 616422895, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019c3e15-d38e-7721-a51f-fdcb9a6f72cf-0', usage_metadata={'input_tokens': 11, 'output_tokens': 23, 'total_tokens': 34})

In [7]:
llm.invoke([HumanMessage("Hello")])

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-02-01T15:00:52.232755566Z', 'done': True, 'done_reason': 'stop', 'total_duration': 418542171, 'load_duration': 112667937, 'prompt_eval_count': 11, 'prompt_eval_duration': 44573610, 'eval_count': 10, 'eval_duration': 254030001, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019c19b8-8fe5-7e31-a13f-9f49b3bc3831-0', usage_metadata={'input_tokens': 11, 'output_tokens': 10, 'total_tokens': 21})

## Streaming

In [9]:
for token in llm.stream([HumanMessage("Hello")]):
    print(token.content, end="|")

Hello|!| How| are| you| today|?| Is| there| something| I| can| help| you| with| or| would| you| like| to| chat|?|||

## Prompt Template

In [12]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from English into {language}"

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

In [13]:
prompt = prompt_template.invoke({"language": "Spanish", "text": "Bye!"})

prompt

ChatPromptValue(messages=[SystemMessage(content='Translate the following from English into Spanish', additional_kwargs={}, response_metadata={}), HumanMessage(content='Bye!', additional_kwargs={}, response_metadata={})])

In [14]:
prompt.to_messages()

[SystemMessage(content='Translate the following from English into Spanish', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Bye!', additional_kwargs={}, response_metadata={})]

In [16]:
response = llm.invoke(prompt)
print(response.content)

¡Hasta luego!


# Semantic search

In [7]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [8]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Elephants are the largest land animals, known for their intelligence and strong social bonds.",
        metadata={"source": "wildlife-doc"},
        id="doc1",
    ),
    Document(
        page_content="Tigers are powerful predators, recognized by their distinctive striped fur.",
        metadata={"source": "wildlife-doc"},
        id="doc2",
    ),
    # Add more documents as needed
]

In [9]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./langchain-exercises-main/docs/example_data/nke-10k-2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

107


In [10]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FO

{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': './langchain-exercises-main/docs/example_data/nke-10k-2023.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}


## Splitting

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

516

In [12]:
print(all_splits[0].page_content)

Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FOR THE FISCAL YEAR ENDED MAY 31, 2023
OR
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FOR THE TRANSITION PERIOD FROM                         TO                         .
Commission File No. 1-10635
NIKE, Inc.
(Exact name of Registrant as specified in its charter)
Oregon 93-0584541
(State or other jurisdiction of incorporation) (IRS Employer Identification No.)
One Bowerman Drive, Beaverton, Oregon 97005-6453
(Address of principal executive offices and zip code)
(503) 671-6453
(Registrant's telephone number, including area code)
SECURITIES REGISTERED PURSUANT TO SECTION 12(B) OF THE ACT:
Class B Common Stock NKE New York Stock Exchange
(Title of each class) (Trading symbol) (Name of each exchange on which registered)


In [13]:
print(all_splits[1].page_content)

SECURITIES REGISTERED PURSUANT TO SECTION 12(B) OF THE ACT:
Class B Common Stock NKE New York Stock Exchange
(Title of each class) (Trading symbol) (Name of each exchange on which registered)
SECURITIES REGISTERED PURSUANT TO SECTION 12(G) OF THE ACT:
NONE
Indicate by check mark: YES NO
• if the registrant is a well-known seasoned issuer, as defined in Rule 405 of the Securities Act. þ ¨
• if the registrant is not required to file reports pursuant to Section 13 or Section 15(d) of the Act. ¨ þ
• whether the registrant (1) has filed all reports required to be filed by Section 13 or 15(d) of the Securities Exchange Act of 1934 during the preceding
12 months (or for such shorter period that the registrant was required to file such reports), and (2) has been subject to such filing requirements for the
past 90 days.
þ ¨
• whether the registrant has submitted electronically every Interactive Data File required to be submitted pursuant to Rule 405 of Regulation S-T


In [18]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="llama3.1:8b",
)

In [19]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 4096

[0.00071356865, -0.016845383, -0.017632933, -0.001921348, 0.00986967, -0.014647882, 0.002095011, -0.02143546, -0.012102062, -0.006204842]


## Vector Store

In [20]:
from langchain_community.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [21]:
ids = vector_store.add_documents(documents=all_splits)

In [22]:
# Return documents based on similarity to a string query

results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

print(results[0])

page_content='• We offer free access to our Sport Centers at our world headquarters for our full-time employees and North America store employees.
• We provide employees free access to mindfulness and meditation resources, as well as live classes through our Sport Centers.
• We provide all employees and their families globally with free and confidential visits with a mental health counselor through a third-party provider and our global
Employee Assistance Program (EAP).
• We provide support to our employees in a variety of ways during times of crisis, including pay continuity under certain circumstances, our natural disaster assistance
program, and ongoing support for challenges related to the COVID-19 pandemic.
• We provide a hybrid work approach for the majority of employees, as well as a Four Week Flex, which provides employees an opportunity to work from a location of
their choice for up to four weeks per year.' metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator'

In [23]:
# Async query
results = await vector_store.asimilarity_search("When was Nike incorporated?")

print(results[0])

page_content='Matthew Friend, Executive Vice President and Chief Financial Officer — Mr. Friend, 45, joined NIKE in 2009 and leads the Company'sfinance, demand & supply management, procurement and global places & services organizations. He joined NIKE as Senior Director ofCorporate Strategy and Development, and was appointed Chief Financial Officer of Emerging Markets in 2011. In 2014, Mr. Friend wasappointed Chief Financial Officer of Global Categories, Product and Functions, and was subsequently appointed Chief Financial Officer ofthe NIKE Brand in 2016. He was also appointed Vice President of Investor Relations in 2019. Mr. Friend was appointed as Executive VicePresident and Chief Financial Officer of NIKE, Inc. in April 2020. Prior to joining NIKE, he worked in the financial industry including rolesas VP of investment banking and mergers and acquisitions at Goldman Sachs and Morgan Stanley.' metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML 

In [24]:
# Return scores
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("What was Nike's revenue in 2023?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)


Score: 0.6052855109859802

page_content='Sales through NIKE Direct 21,308 18,726 14 % 20 % 16,370 14 % 15 %
Global Brand Divisions 58 102 -43 % -43 % 25 308 % 302 %
TOTAL NIKE BRAND REVENUES $ 48,763 $ 44,436 10 % 16 %$ 42,293 5 % 6 %
NIKE Brand Revenues on a Wholesale Equivalent Basis :
Sales to Wholesale Customers $ 27,397 $ 25,608 7 % 14 %$ 25,898 -1 % -1 %
Sales from our Wholesale Operations to NIKE Direct Operations 12,730 10,543 21 % 27 % 9,872 7 % 7 %
TOTAL NIKE BRAND WHOLESALE EQUIVALENT REVENUES $ 40,127 $ 36,151 11 % 18 %$ 35,770 1 % 1 %
NIKE Brand Wholesale Equivalent Revenues by:
Men's $ 20,733 $ 18,797 10 % 17 %$ 18,391 2 % 3 %
Women's 8,606 8,273 4 % 11 % 8,225 1 % 1 %
NIKE Kids' 5,038 4,874 3 % 10 % 4,882 0 % 0 %
Jordan Brand 6,589 5,122 29 % 35 % 4,780 7 % 7 %
Others (839) (915) 8 % -3 % (508) -80 % -79 %
TOTAL NIKE BRAND WHOLESALE EQUIVALENT REVENUES $ 40,127 $ 36,151 11 % 18 %$ 35,770 1 % 1 %' metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'ED

In [25]:
# Return documents based on similarity to an embedded query
embedding = embeddings.embed_query("How were Nike's margins impacted in 2023?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='In recent years, uncertain global and regional economic and political conditions have affected international trade and increased protectionist actions around the
world. These trends are affecting many global manufacturing and service sectors, and the footwear and apparel industries, as a whole, are not immune. Companies in our
industry are facing trade protectionism in many different regions, and, in nearly all cases, we are working together with industry groups to address trade issues and reduce
the impact to the industry, while observing applicable competition laws. Notwithstanding our efforts, protectionist measures have resulted in increases in the cost of our
products, and additional measures, if implemented, could adversely affect sales and/or profitability for NIKE, as well as the imported footwear and apparel industry as a
whole.' metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:2

## Vector store but with google gemmini

Because tiny quantized ollama is mediocre

In [60]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    api_key =os.getenv("google_api_key")
)

vector_1 = embeddings.embed_query(all_splits[0].page_content)
print(f"Generated vector of length {len(vector_1)}\n")

vector_store = InMemoryVectorStore(embeddings)

Generated vector of length 3072



In [ ]:
# Also because free gemini tier is api limited i had to split it, but at least it performs well
import time
import os
if os.path.exists("./pickled_objects/practice-vector-store-gemini001.dump"):
  vector_store = vector_store.load("pickled_objects/practice-vector-store-gemini001.dump",embeddings)
  #noIds for now
else:
  ids = []
  for i in range(0,len(all_splits),100):
    if i!=0:
      time.sleep(60)
    endi = min(i+100,len(all_splits))
    print(i,endi-1)
    ids += vector_store.add_documents(documents=all_splits[i:endi])
  vector_store.dump("pickled_objects/practice-vector-store-gemini001.dump")

In [68]:
# Return documents based on similarity to a string query

results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

print(results[0])

page_content='operations. We also lease an office complex in Shanghai, China, our headquarters for our Greater China geography, occupied by employees focused on implementing our
wholesale, NIKE Direct and merchandising strategies in the region, among other functions.
In the United States, NIKE has eight significant distribution centers. Five are located in or near Memphis, Tennessee, two of which are owned and three of which are
leased. Two other distribution centers, one located in Indianapolis, Indiana and one located in Dayton, Tennessee, are leased and operated by third-party logistics
providers. One distribution center for Converse is located in Ontario, California, which is leased. NIKE has a number of distribution facilities outside the United States,
some of which are leased and operated by third-party logistics providers. The most significant distribution facilities outside the United States are located in Laakdal,' metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 

In [69]:
# Async query
results = await vector_store.asimilarity_search("When was Nike incorporated?")

print(results[0])

page_content='Table of Contents
PART I
ITEM 1. BUSINESS
GENERAL
NIKE, Inc. was incorporated in 1967 under the laws of the State of Oregon. As used in this Annual Report on Form 10-K (this "Annual Report"), the terms "we," "us," "our,"
"NIKE" and the "Company" refer to NIKE, Inc. and its predecessors, subsidiaries and affiliates, collectively, unless the context indicates otherwise.
Our principal business activity is the design, development and worldwide marketing and selling of athletic footwear, apparel, equipment, accessories and services. NIKE is
the largest seller of athletic footwear and apparel in the world. We sell our products through NIKE Direct operations, which are comprised of both NIKE-owned retail stores
and sales through our digital platforms (also referred to as "NIKE Brand Digital"), to retail accounts and to a mix of independent distributors, licensees and sales' metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'cr

In [70]:
# Return scores
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("What was Nike's revenue in 2023?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)


Score: 0.770986831835312

page_content='Table of Contents
FISCAL 2023 NIKE BRAND REVENUE HIGHLIGHTSThe following tables present NIKE Brand revenues disaggregated by reportable operating segment, distribution channel and major product line:
FISCAL 2023 COMPARED TO FISCAL 2022
• NIKE, Inc. Revenues were $51.2 billion in fiscal 2023, which increased 10% and 16% compared to fiscal 2022 on a reported and currency-neutral basis, respectively.
The increase was due to higher revenues in North America, Europe, Middle East & Africa ("EMEA"), APLA and Greater China, which contributed approximately 7, 6,
2 and 1 percentage points to NIKE, Inc. Revenues, respectively.
• NIKE Brand revenues, which represented over 90% of NIKE, Inc. Revenues, increased 10% and 16% on a reported and currency-neutral basis, respectively. This
increase was primarily due to higher revenues in Men's, the Jordan Brand, Women's and Kids' which grew 17%, 35%,11% and 10%, respectively, on a wholesale
equivalent basis.' metada

In [71]:
# Return documents based on similarity to an embedded query
embedding = embeddings.embed_query("How were Nike's margins impacted in 2023?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='Table of Contents
GROSS MARGIN
FISCAL 2023 COMPARED TO FISCAL 2022
For fiscal 2023, our consolidated gross profit increased 4% to $22,292 million compared to $21,479 million for fiscal 2022. Gross margin decreased 250 basis points to
43.5% for fiscal 2023 compared to 46.0% for fiscal 2022 due to the following:
*Wholesale equivalent
The decrease in gross margin for fiscal 2023 was primarily due to:
• Higher NIKE Brand product costs, on a wholesale equivalent basis, primarily due to higher input costs and elevated inbound freight and logistics costs as well as
product mix;
• Lower margin in our NIKE Direct business, driven by higher promotional activity to liquidate inventory in the current period compared to lower promotional activity in
the prior period resulting from lower available inventory supply;
• Unfavorable changes in net foreign currency exchange rates, including hedges; and
• Lower off-price margin, on a wholesale equivalent basis.
This was partially offset by:'

In [72]:
len(results)

4